# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [21]:
!wget https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv

import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')

df = df.dropna()

df["down_trend"] = (df["trend_direction"] == "down").astype(int)


--2026-08-26 04:00:34--  https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6727670 (6.4M) [text/plain]
Saving to: ‘content_refresh_anonymized.csv.4’

content_refresh_ano 100%[===================>]   6.42M  --.-KB/s    in 0.06s   

2026-08-26 04:00:34 (113 MB/s) - ‘content_refresh_anonymized.csv.4’ saved [6727670/6727670]



In [22]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,precision_score,recall_score
from sklearn.linear_model import LogisticRegression


features =[
       'scroll_rate','clicks_90d', 'pageviews_90d','cpc','sessions_90d', 'users_90d', 'engagement_rate','search_volume']

new_X = df[features]
new_y = df["down_trend"]

new_X_train, new_X_test, new_y_train, new_y_test = train_test_split(new_X, new_y, test_size=0.3, random_state=42, stratify=new_y)

In [23]:

new_y_train.shape

(5145,)

In [24]:

new_y_test.shape

(2205,)

In [25]:

new_X_train.shape

(5145, 8)

In [26]:
new_X_test.shape

(2205, 8)

In [27]:
new_model = LogisticRegression(max_iter=1000)

new_model.fit(new_X_train, new_y_train)


LogisticRegression(max_iter=1000)

In [28]:
new_test_results = new_X_test.copy()
new_test_results["model_predictions"] = new_model.predict(new_X_test)
new_test_results["actual_values"] = new_y_test


In [35]:
new_test_results["down_probability"] = new_model.predict_proba(new_X_test)[:, 1]
ranked_queue = new_test_results.sort_values(
    "down_probability",
    ascending=False
).copy()

ranked_queue.head()

,scroll_rate,clicks_90d,pageviews_90d,cpc,sessions_90d,users_90d,engagement_rate,search_volume,model_predictions,actual_values,down_probability
17216,9.13,36,1227,0.00,1006,1009,0.50,0.0,1,1,0.941496
24766,300.00,0,1,0.00,4,4,0.00,0.0,1,1,0.924865
661,0.88,0,1134,0.39,1059,935,0.94,260.0,1,1,0.902981
5971,200.00,0,1,0.00,3,3,0.00,0.0,1,1,0.872063
22739,200.00,0,1,0.00,2,2,0.00,0.0,1,0,0.871307


In [36]:
ranked_queue["action"] = ranked_queue["model_predictions"].map({
    1: "Refresh",
    0: "Monitoring"
})

In [38]:
ranked_queue["reason_code"] = ranked_queue["model_predictions"].map({
    1: "Predicted_Downtrend",
    0: "No_Downtrend_Signal"
})

In [39]:
ranked_queue.head()

,scroll_rate,clicks_90d,pageviews_90d,cpc,sessions_90d,users_90d,engagement_rate,search_volume,model_predictions,actual_values,down_probability,action,reason_code
17216,9.13,36,1227,0.00,1006,1009,0.50,0.0,1,1,0.941496,Refresh,Predicted_Downtrend
24766,300.00,0,1,0.00,4,4,0.00,0.0,1,1,0.924865,Refresh,Predicted_Downtrend
661,0.88,0,1134,0.39,1059,935,0.94,260.0,1,1,0.902981,Refresh,Predicted_Downtrend
5971,200.00,0,1,0.00,3,3,0.00,0.0,1,1,0.872063,Refresh,Predicted_Downtrend
22739,200.00,0,1,0.00,2,2,0.00,0.0,1,0,0.871307,Refresh,Predicted_Downtrend


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Who uses it:
Content/SEO teams use the model to identify pages that are likely candidates for refresh or further review.

What it is used for:
The model ranks pages by their estimated likelihood of experiencing a downward performance trend, helping the team decide which pages should be investigated first. The output is a prioritization tool, not an automatic decision-maker


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

The SEO team should review the context features of the data first before taking action as they provide more insight.

The system should never automatically delete or unpublish pages, rewrite content or make other irreversible SEO decisions. It should also never claim that a particular factor caused a performance decline simply because the model identified an association. The model's role is to determine which pages should be reviewed first, while a human remains responsible for determining what action should actually be taken.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

If predictions performance started to decline then we would know the model need reviewing based on the new data shift. New data affects the model significantly and the goal is to make this model usable with new data and the only way is to maintain and re-evaluate the model with the new data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.